# Détection Minérale Spécifique (Anomalies)

Ce notebook identifie les anomalies spectrales pouvant indiquer la présence de minerais (coltan, or, cuivre) en utilisant le Z-score sur les caractéristiques Prithvi.

In [ ]:
!pip install geemap earthengine-api rasterio terratorch torch matplotlib -q
import ee, geemap, torch, rasterio, os
import numpy as np
import matplotlib.pyplot as plt
from terratorch import BACKBONE_REGISTRY

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([14.8, -5.2, 15.2, -4.8])
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate('2023-01-01', '2023-12-31').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
image = collection.median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('input.tif') as src: img = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats.numpy().reshape(-1, feats.shape[-1])
z_scores = np.abs((feats_np - np.mean(feats_np, 0)) / (np.std(feats_np, 0) + 1e-8))
anomaly_map = np.max(z_scores, 1).reshape(int(np.sqrt(feats_np.shape[0]-1)), -1)

plt.imshow(anomaly_map, cmap='magma')
plt.title("Anomalies Minérales (Z-Score Prithvi)")
plt.colorbar()
plt.show()